## What is the Random Walker?

The RandomWalker brick solves diffusion and particle transport problems with the density method: a large group of random walkers hops between discrete states, and the number of walkers in each state over time is the solution. The walk follows a Markov chain

$$\rho_{t+1} = \rho_t P$$

where P is the transition matrix (P[i,j] is the probability that a walker steps from state i to state j) and rho_t is the walker density at model step t. The walkers are carried entirely by spiking neurons, and counting their readout spikes recovers the density.

In this notebook we reproduce the two-state angular flux benchmark (Fig. 3c of the paper below). Particles travel in a positive or negative direction, scatter between the two, and are occasionally absorbed. We compare the spiking result against the exact analytic solution.

The paper that inspired this Fugu implementation is the following:

[1] J. D. Smith et al., "Neuromorphic scaling advantages for energy-efficient random walk computations," Nature Electronics 5, 102-112 (2022). DOI 10.1038/s41928-021-00705-7 (Supplementary Note 3.8).

## Setup

Import the libraries and define the problem. The initial fluxes are 5 (positive) and 3 (negative). We seed walkers in the same 5:3 ratio, 300 and 180, so one walker is worth 1/60 of a flux unit.

The transition matrix comes from the physics: in one time step a particle either keeps its direction, scatters into a uniformly random direction, or is absorbed. State 2 is the absorbed state; its row is all zeros, which tells the brick to discard walkers that land there.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fugu.scaffold import Scaffold
from fugu.bricks import RandomWalker
from fugu.backends import snn_Backend

np.random.seed(0)   # the walk's coin flips draw from this RNG

SigS = 8.0    # scattering rate
SigA = 2.0    # absorption rate
dt   = 0.005  # physical time per model step
i_pos = 5.0   # initial positive flux
i_neg = 3.0   # initial negative flux

walkers = {0: 300, 1: 180}         # seeded in the 5:3 ratio of the fluxes
flux_per_walker = (i_pos + i_neg) / 480   # one walker = 1/60 flux unit
neural_timesteps = 80000           # simulator tick budget, takes about a minute

# probabilities for one time step
p_event   = (SigS + SigA) * dt * np.exp(-(SigA + SigS) * dt)   # chance of an event
p_scatter = p_event * SigS / (SigA + SigS)
p_absorb  = p_event * SigA / (SigA + SigS)
p_stay    = 1 - p_event

P = np.array([
    [p_stay + p_scatter / 2,  p_scatter / 2,           p_absorb],
    [p_scatter / 2,           p_stay + p_scatter / 2,  p_absorb],
    [0.0,                     0.0,                     0.0     ],
])

print('Transition matrix P:')
print(np.round(P, 4))

## Compute Ground Truth

The benchmark has an exact closed-form solution with two exponential modes: the total flux decays like exp(-SigA t) and the imbalance between the two directions decays like exp(-(SigA+SigS) t). This tells us the curves the spiking walk should recover.

In [ ]:
def analytic_flux(t):
    total     = (i_pos + i_neg) * np.exp(-SigA * t)
    imbalance = (i_pos - i_neg) * np.exp(-(SigA + SigS) * t)
    positive = (total + imbalance) / 2
    negative = (total - imbalance) / 2
    return positive, negative

pos0, neg0 = analytic_flux(0.0)
print(f'Analytic flux at t=0: ({pos0:.1f}, {neg0:.1f})')

## Build the Neural Circuit

Create a Scaffold and add the RandomWalker brick. The brick builds the whole circuit from the transition matrix.

Inside, each state gets a small team of LIF neurons. A counter stores the state's walkers as negative potential, a self-firing generator pays them out one per tick, probabilistic gate neurons flip the transition coins so exactly one exit opens per walker, a buffer holds arriving walkers until the step boundary, and a readout emits one recordable spike per walker. Four global controller neurons keep all states in lockstep, so one full pay-out and transfer cycle is one model step.

RandomWalker is a source brick: the matrix and the initial walkers go straight into the constructor, no input brick is needed, and output=True tells the backend to record the readout neurons.

In [ ]:
scaffold = Scaffold()
scaffold.add_brick(RandomWalker(P, walkers, timesteps=neural_timesteps), output=True)
scaffold.lay_bricks()

## Compile to the SNN Backend

Compile the scaffold to Fugu's stock snn backend. The random walker needs no specialized backend.

In [ ]:
backend = snn_Backend()
backend.compile(scaffold)
print('Compiled:', len(scaffold.graph.nodes), 'neurons,', len(scaffold.graph.edges), 'synapses')

## Run the Simulation

Run the spiking network. The result is a table with one row per readout spike, with columns (time, neuron_number). Each spike is one walker being counted in some state. This takes about a minute.

In [ ]:
spikes = backend.run(neural_timesteps).copy()
print(len(spikes), 'readout spikes recorded')

## Analyze Results

The spikes are decoded in one pass. First we look up which readout neuron reports which state (each one carries its state number in its index tag). Then we read the spike log in time order: spikes come in bursts, one burst per model step, with silent gaps in between while walkers transfer between states. So whenever the time jumps by more than one tick, a new step begins. Within a step we tally the spikes per state; the tally is the walker count, and multiplying by flux_per_walker turns it into flux.

In [ ]:
# which readout neuron belongs to which state
state_of = {}
for name, data in scaffold.graph.nodes.data():
    if 'readout' in str(name):
        state_of[data['neuron_number']] = data['index'][0]

# read the spike log in time order: a gap of more than 1 tick starts a new
# model step, and every spike adds one walker to its state's tally
counts = []
last_time = -1
for time, neuron in zip(spikes['time'], spikes['neuron_number']):
    if time - last_time > 1:
        counts.append([0, 0, 0])        # a new step: [positive, negative, absorbed]
    counts[-1][state_of[neuron]] += 1
    last_time = time

counts = counts[:-2]                    # the run stops mid-step, drop the last two to be safe
n_steps = len(counts)
flux = np.array(counts).T[:2] * flux_per_walker   # rows: positive, negative

print('Model steps captured:', n_steps, '(physical time reached:', round(n_steps * dt, 2), ')')
print('Flux at t=0:', flux[0, 0], ',', flux[1, 0])

## Plot the Results

The spiking flux (red and blue) against the exact analytic solution (dashed). This is the Fig. 3c reproduction.

In [ ]:
tarray = np.arange(n_steps) * dt
apos, aneg = analytic_flux(tarray)

plt.figure(figsize=(7, 4))
plt.plot(tarray, apos, 'k--', label='Analytic')
plt.plot(tarray, aneg, 'k--')
plt.plot(tarray, flux[0], 'r', label='Positive')
plt.plot(tarray, flux[1], 'b', label='Negative')
plt.xlabel('Time')
plt.ylabel('Flux')
plt.legend()
plt.title('Two-state angular flux: RandomWalker brick vs analytic')
plt.show()

## Ground Truth Comparison

Two checks. Conservation: at step 0 every seeded walker is paid out exactly once, so the flux must start at exactly (5, 3). Any deviation means walkers were lost or duplicated. Accuracy: the average deviation from the analytic curves should stay within Monte Carlo noise, which is about 0.07 flux units for 480 walkers. A broken circuit scores 2 to 4, so the 0.5 gate separates them cleanly.

In [ ]:
conserved = abs(flux[0, 0] - i_pos) < 0.5 and abs(flux[1, 0] - i_neg) < 0.5
print(f'Flux at t=0: ({flux[0, 0]:.2f}, {flux[1, 0]:.2f})   expected: ({i_pos}, {i_neg})')

w = min(400, n_steps)
apos_w, aneg_w = analytic_flux(np.arange(w) * dt)
err_pos = np.mean(np.abs(flux[0, :w] - apos_w))
err_neg = np.mean(np.abs(flux[1, :w] - aneg_w))
error = (err_pos + err_neg) / 2
print(f'Mean difference from the analytic curves over the first {w} steps: {error:.4f}')

print('\nMatch:', bool(conserved and error < 0.5))

## Summary Table

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Walkers seeded (+, -)', 'Neural timesteps', 'Model steps captured',
               'Physical time reached', 'Flux at t=0 (expect 5, 3)',
               'Mean error vs analytic'],
    'Value': [f'{walkers[0]}, {walkers[1]}', neural_timesteps, n_steps,
              f'{n_steps * dt:.2f}', f'{flux[0, 0]:.2f}, {flux[1, 0]:.2f}',
              f'{error:.4f}'],
})

print('=' * 70)
print('RANDOM WALKER RESULTS SUMMARY')
print('=' * 70)
print(summary.to_string(index=False))
print('=' * 70)

if conserved and error < 0.5:
    print('\nSUCCESS: the spiking random walk reproduces the analytic solution')